In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm

In [ ]:
def merge_dataframes(chembl, bindingdb):

    dfs = [chembl, bindingdb]
    merged = pd.concat(dfs, ignore_index=True)

    # Sort df by smiles so all the same smiles are together
    merged = merged.sort_values('smiles').reset_index(drop=True)

    # Statistics
    print(f"\nNumber of rows in merged dataframe: {len(merged)}")
    print(f"Number of unique SMILES-target pairs in merged dataframe: {merged.groupby(['smiles', 'uniprot_id']).ngroups}")
    print(f"Number of unique SMILES in merged dataframe: {merged['smiles'].nunique()}")

    targets_per_smiles = merged.groupby('smiles')['uniprot_id'].nunique()
    print(f"Average number of targets per SMILES: {targets_per_smiles.mean():.2f}")
    measurements_per_smiles_target = merged.groupby(['smiles', 'uniprot_id']).size()
    print(f"Average number of measurements per SMILES and target: {measurements_per_smiles_target.mean():.2f}")

    print("\n")
    print(merged['bioactivity_relation'].value_counts().to_string())

    greater_than_data = merged[merged['bioactivity_relation'].isin(['>', '>='])]
    print(f"Number of measurements with > relation and bioactivity < 10000: {len(greater_than_data[greater_than_data['bioactivity'] < 10000])}")
    equal_data = merged[merged['bioactivity_relation'] == '=']
    print(f"Number of measurements with = relation and bioactivity > 10000: {len(equal_data[equal_data['bioactivity'] > 10000])}")

    print("\n")
    print(merged['activity_comment'].unique())

    return merged

def extract_unique_smiles(df):

    combined = pd.concat([df["drug"], df["metabolite"]], ignore_index=True)
    unique_smiles = combined.drop_duplicates()
    df = unique_smiles.to_frame(name="smiles")

    return df

LAGOM = pd.read_csv("data/extended_LAGOM_dataset.csv")
LAGOM_smiles = extract_unique_smiles(LAGOM)
chembl = pd.read_csv("data/chembl_smiles.csv")
bindingdb = pd.read_csv("data/bindingdb_smiles.csv")

print(f'Number of bioactivity measurements in ChEMBL: {len(chembl)}')
print(f'Number of bioactivity measurements in BindingDB: {len(bindingdb)}')

print(f'Number of unique reactions in LAGOM: {len(LAGOM)}')
print(f"Number of unique SMILES in LAGOM: {LAGOM_smiles['smiles'].nunique()}")
print(f"Number of unique SMILES in ChEMBL: {chembl['smiles'].nunique()}")
print(f"Number of unique SMILES in BindingDB: {bindingdb['smiles'].nunique()}")

merged = merge_dataframes(chembl, bindingdb)
merged.to_csv("inbetween_data/merged_smiles.csv", index=False)

Number of bioactivity measurements in ChEMBL: 23474
Number of bioactivity measurements in BindingDB: 32940
Number of unique reactions in LAGOM: 4191
Number of unique SMILES in LAGOM: 5322
Number of unique SMILES in ChEMBL: 1035
Number of unique SMILES in BindingDB: 1169

Number of rows in merged dataframe: 56414
Number of unique SMILES-target pairs in merged dataframe: 15572
Number of unique SMILES in merged dataframe: 1287
Average number of targets per SMILES: 12.10
Average number of measurements per SMILES and target: 3.62


bioactivity_relation
=     40122
>     15781
<       454
>=       18
<=        8
~         4
Number of measurements with > relation and bioactivity < 10000: 710
Number of measurements with = relation and bioactivity > 10000: 5365


[nan 'Not Active' 'Cytochrome P450 mechanistic inhibitor: PMID 11907170'
 'Cytochrome P450 mechanistic inhibitor'
 'Cytochrome P450 mechanistic inhibitor: PMID 11805216' '486914' '486913'
 '302688' '301101' '301098' '302691' '234363' '

In [ ]:
def filter_bioactivity(df):

    # Standardize 'bioactivity_relation'
    df['bioactivity_relation'] = df['bioactivity_relation'].replace('>=', '>')
    df['bioactivity_relation'] = df['bioactivity_relation'].replace('<=', '<') 

    removed_rows = pd.DataFrame()

    # Remove rows that misses either bioactivity value, relation or type
    mask = df['bioactivity'].isna() | df['bioactivity_relation'].isna() | df['bioactivity_type'].isna()
    discarded = df[mask].copy()
    discarded['reason'] = 'missing bioactivity value, relation or type'
    removed_rows = pd.concat([removed_rows, discarded], ignore_index=True)
    df = df[~mask]

    # Remove rows with 'Uncertain' in 'activity_comment'
    mask = df['activity_comment'] == 'Uncertain'
    discarded = df[mask].copy()
    discarded['reason'] = 'activity_comment Uncertain'
    removed_rows = pd.concat([removed_rows, discarded], ignore_index=True)
    df = df[~mask]

    # Remove rows with '~' in 'bioactivity_relation'
    mask = df['bioactivity_relation'] == '~'
    discarded = df[mask].copy()
    discarded['reason'] = 'bioactivity_relation ~'
    removed_rows = pd.concat([removed_rows, discarded], ignore_index=True)
    df = df[~mask]

    # Remove rows with '<' in 'bioactivity_relation'
    mask = df['bioactivity_relation'] == '<'
    discarded = df[mask].copy()
    discarded['reason'] = 'bioactivity_relation <'
    removed_rows = pd.concat([removed_rows, discarded], ignore_index=True)
    df = df[~mask]

    # Keep only rows with bioactivity values of interest
    mask = ((df['bioactivity_relation'] == '=') & (df['bioactivity'] > 10000)) | ((df['bioactivity_relation'] == '>') & (df['bioactivity'] < 10000))
    discarded = df[mask].copy()
    discarded['reason'] = 'bioactivity out of range'
    removed_rows = pd.concat([removed_rows, discarded], ignore_index=True)
    df = df[~mask]

    # If a smiles-target-type pair has both '=' and '>' measurements, investigate further
    relation_counts = df.groupby(['smiles', 'uniprot_id', 'bioactivity_type'])['bioactivity_relation'].nunique()
    conflicting_groups = relation_counts[relation_counts > 1].index
    masks_to_remove = []
    for group in conflicting_groups:
        smiles, uniprot, bio_type = group
        mask_equal = (
            (df['smiles'] == smiles) & 
            (df['uniprot_id'] == uniprot) & 
            (df['bioactivity_type'] == bio_type) &
            (df['bioactivity_relation'] == '=')
        )
        mask_greater = (
            (df['smiles'] == smiles) & 
            (df['uniprot_id'] == uniprot) & 
            (df['bioactivity_type'] == bio_type) &
            (df['bioactivity_relation'] == '>')
        )
        if mask_equal.any() and mask_greater.any():
            min_equal = df.loc[mask_equal, 'bioactivity'].min()
            min_greater = df.loc[mask_greater, 'bioactivity'].min()

            if min_equal >= min_greater:
                # Remove '>' rows
                masks_to_remove.append(mask_greater)
            else:
                # Remove both
                masks_to_remove.append(mask_equal | mask_greater)

    if masks_to_remove:
        mask = masks_to_remove[0]
        for m in masks_to_remove[1:]:
            mask |= m

    discarded = df[mask].copy()
    discarded['reason'] = "conflicting bioactivity_relation ('=' and '>')"
    removed_rows = pd.concat([removed_rows, discarded], ignore_index=True)
    df = df[~mask]

    removed_rows.to_csv('inbetween_data/removed_rows_filtering.csv', index=False)

    # Identify and update source for mixed entries
    source_groups = df.groupby(['smiles', 'uniprot_id', 'bioactivity_type', 'bioactivity_relation'])['source'].nunique()
    overlapping_groups = source_groups[source_groups > 1].index
    for group in overlapping_groups:
        smiles, uniprot, bio_type, bio_rel = group
        mask = (
            (df['smiles'] == smiles) & 
            (df['uniprot_id'] == uniprot) & 
            (df['bioactivity_type'] == bio_type) &
            (df['bioactivity_relation'] == bio_rel)
        )
        df.loc[mask, 'source'] = 'Both'

    # Drop unnecessary columns and reset index
    df = df.drop(columns=['chembl_id', 'target_id', 'activity_comment'])
    df = df.reset_index(drop=True) 
    
    # Statistics
    print(f"Number of rows after filtering: {len(df)}")
    print(f"Number of unique SMILES in merged dataframe: {df['smiles'].nunique()}")
    targets_per_smiles = df.groupby('smiles')['uniprot_id'].nunique()
    print(f"Average number of targets per SMILES: {targets_per_smiles.mean():.2f}")
    measurements_per_smiles_target = df.groupby(['smiles', 'uniprot_id']).size()
    print(f"Average number of measurements per SMILES and target: {measurements_per_smiles_target.mean():.2f}")

    print(f"Rows removed: {len(removed_rows)}")
    print(removed_rows['reason'].value_counts().to_string())

    return df

merged = pd.read_csv("inbetween_data/merged_smiles.csv")
merged_filtered = filter_bioactivity(merged)
merged_filtered.to_csv('inbetween_data/merged_smiles_filtered.csv', index=False)

Number of rows after filtering: 44407
Number of unique SMILES in merged dataframe: 1163
Average number of targets per SMILES: 11.38
Average number of measurements per SMILES and target: 3.36
Rows removed: 12007
reason
bioactivity out of range                          6075
conflicting bioactivity_relation ('=' and '>')    5439
bioactivity_relation <                             462
missing bioactivity value, relation or type         27
bioactivity_relation ~                               4


In [ ]:
def aggregate_data(df):

    # Convert bioactivity to p-bioactivity
    df['p_bioactivity'] = np.nan
    valid_bioactivity = df['bioactivity'] > 0
    df.loc[valid_bioactivity, 'p_bioactivity'] = -np.log10(df.loc[valid_bioactivity, 'bioactivity'] * 1e-9)

    # Aggregate data
    df_stats = df.groupby(
        ['smiles', 'uniprot_id', 'bioactivity_relation', 'bioactivity_type', 'source']
    ).agg(
        num_entries=('smiles', 'count'),
        p_mean=('p_bioactivity', lambda x: round(x.mean(), 2)),
        p_std=('p_bioactivity', lambda x: round(x.std(), 2))
    ).reset_index()

    # How many with std > 1
    high_std = df_stats[(df_stats['p_std'] >= 1) & (df_stats['bioactivity_relation'] == '>')]
    print(f"Number of SMILES-target pairs with std > 1 and relation '>': {len(high_std)}")
    high_std = df_stats[(df_stats['p_std'] >= 1) & (df_stats['bioactivity_relation'] == '=')]
    print(f"Number of SMILES-target pairs with std > 1 and relation '=': {len(high_std)}")

    # Keep only rows where std < 1 for '=' relation, keep all rows for '>' relation
    mask = (df_stats['p_std'] >= 1) & (df_stats['bioactivity_relation'] == '=')
    discarded = df_stats[mask].copy()
    discarded['reason'] = 'std>1 for = relation'
    discarded.to_csv('inbetween_data/removed_rows_aggregating.csv', index=False)
    final_df = df_stats[~mask].reset_index(drop=True)

    # Replace value of > relation with the maximum p_bioactivity in df
    for idx, row in final_df[final_df['bioactivity_relation'] == '>'].iterrows():
        smiles = row['smiles']
        uniprot_id = row['uniprot_id']
        bioactivity_type = row['bioactivity_type']
        max_p = df[
            (df['smiles'] == smiles) &
            (df['uniprot_id'] == uniprot_id) &
            (df['bioactivity_type'] == bioactivity_type) &
            (df['bioactivity_relation'] == '>')
        ]['p_bioactivity'].max()
        final_df.at[idx, 'p_mean'] = round(max_p, 2)

    # Rename and reorder columns
    final_df = final_df.rename(columns={
        'p_mean': 'bioactivity',
    })
    new_column_order = [
        'smiles',
        'bioactivity',
        'bioactivity_type',
        'bioactivity_relation',
        'uniprot_id',
        'source',
        'num_entries'
    ]
    final_df = final_df[new_column_order]

    # Replace every > with < in bioactivity_relation
    final_df['bioactivity_relation'] = final_df['bioactivity_relation'].replace('>', '<')

    # Add a p in front of bioactivity_type
    final_df['bioactivity_type'] = 'p' + final_df['bioactivity_type']

    # Statistics
    print(f"Number of rows after aggregation: {len(final_df)}")
    print(f"Number of unique SMILES in merged dataframe: {final_df['smiles'].nunique()}")
    targets_per_smiles = final_df.groupby('smiles')['uniprot_id'].nunique()
    print(f"Average number of targets per SMILES: {targets_per_smiles.mean():.2f}")
    measurements_per_smiles_target = final_df.groupby(['smiles', 'uniprot_id']).size()
    print(f"Average number of measurements per SMILES and target: {measurements_per_smiles_target.mean():.2f}")
    print(f"Rows lost to std > 1: {len(discarded)}")

    return final_df

merged_filtered = pd.read_csv("inbetween_data/merged_smiles_filtered.csv")
aggregated_df = aggregate_data(merged_filtered) # approx 2 min
aggregated_df.to_csv("inbetween_data/merged_smiles_aggregated.csv", index=False)

Number of SMILES-target pairs with std > 1 and relation '>': 0
Number of SMILES-target pairs with std > 1 and relation '=': 193
Number of rows after aggregation: 14201
Number of unique SMILES in merged dataframe: 1154
Average number of targets per SMILES: 11.40
Average number of measurements per SMILES and target: 1.08
Rows lost to std > 1: 193


In [ ]:
def match_reactions(df, reactions):
    
    all_matches = []
    for _, reaction in tqdm(reactions.iterrows(), desc="Matching reactions" , total=len(reactions)):
        drug = reaction['drug']
        metabolite = reaction['metabolite']
        
        # Find all bioactivity entries for these drugs and metabolites
        drug_data = df[df['smiles'] == drug]
        metabolite_data = df[df['smiles'] == metabolite]

        # Find matches where UniProt ID and measurement type is the same
        for _, d_row in drug_data.iterrows():
            for _, m_row in metabolite_data.iterrows():
                if d_row['uniprot_id'] == m_row['uniprot_id'] and d_row['bioactivity_type'] == m_row['bioactivity_type']:
                    match = {
                        'drug': drug,
                        'metabolite': metabolite,
                        'uniprot_id': d_row['uniprot_id'],
                        'bioactivity_type': d_row['bioactivity_type'],
                        'bioactivity_drug': d_row['bioactivity'],
                        'bioactivity_metabolite': m_row['bioactivity'],
                        'relation_drug': d_row['bioactivity_relation'],
                        'relation_metabolite': m_row['bioactivity_relation'],
                        'source_drug': d_row['source'],
                        'source_metabolite': m_row['source']
                    }
                    all_matches.append(match)
    
    matches_df = pd.DataFrame(all_matches)

    # Statistics
    print(f"Total matches: {len(matches_df)}")
    print(f"Unique reaction pairs: {matches_df[['drug', 'metabolite']].drop_duplicates().shape[0]}")
    print(f"Unique targets: {matches_df['uniprot_id'].nunique()}")

    # Count how many unique smiles there are in drug and metabolite jointly
    unique_smiles = pd.concat([matches_df['drug'], matches_df['metabolite']]).nunique()
    print(f"Unique SMILES: {unique_smiles}")

    print(matches_df['bioactivity_type'].value_counts().to_string())

    targets_per_smiles = matches_df.groupby(['drug', 'metabolite'])['uniprot_id'].nunique()
    print(f"Average number of targets per reaction: {targets_per_smiles.mean():.2f}")

    return matches_df
    
aggregated_df = pd.read_csv("inbetween_data/merged_smiles_aggregated.csv")
LAGOM_reactions = pd.read_csv('data/extended_LAGOM_dataset.csv')
reactions = match_reactions(aggregated_df, LAGOM_reactions)
reactions.to_csv("inbetween_data/matched_reactions.csv", index=False)

Matching reactions: 100%|██████████| 4191/4191 [00:12<00:00, 329.52it/s]

Total matches: 878
Unique reaction pairs: 236
Unique targets: 419
Unique SMILES: 372
bioactivity_type
pKi      307
pKd      289
pIC50    282
Average number of targets per reaction: 3.60


In [ ]:
# --- DUPLICATES ---
df = pd.read_csv("inbetween_data/matched_reactions.csv")

# Get the duplicates where drug, metabolite, unique_id are the same
df_duplicates = df[df.duplicated(subset=['drug', 'metabolite', 'uniprot_id'], keep=False)]
print(f"Number of duplicate reactions (same drug, metabolite, uniprot_id): {len(df_duplicates)}")
df_duplicates.to_csv('inbetween_data/duplicates.csv', index=False)

idx_to_remove = []
for name, group in df_duplicates.groupby(['drug', 'metabolite', 'uniprot_id'], as_index=False):

    # Keep the row with bioactivity type pKi > pIC50 > pKd
    row_to_keep = None
    if len(group[group['bioactivity_type'] == 'pKi']) > 0:
        row_to_keep = group[group['bioactivity_type'] == 'pKi']
    elif len(group[group['bioactivity_type'] == 'pIC50']) > 0:
        row_to_keep = group[group['bioactivity_type'] == 'pIC50']
    elif len(group[group['bioactivity_type'] == 'pKd']) > 0:
        row_to_keep = group[group['bioactivity_type'] == 'pKd']

    if row_to_keep is not None:
        idxs = group.index.tolist()
        idx_to_keep = row_to_keep.index[0]
        idxs.remove(idx_to_keep)
        idx_to_remove.extend(idxs)

print(f"Indices to remove: {idx_to_remove}")

df_cleaned = df.drop(index=idx_to_remove).reset_index(drop=True)
df_cleaned.to_csv('inbetween_data/matched_reactions_deduplicated.csv', index=False)
print(f"Number of reactions after removing duplicates: {len(df_cleaned)}")

Number of duplicate reactions (same drug, metabolite, uniprot_id): 57
Indices to remove: [168, 112, 113, 127, 152, 153, 265, 286, 677, 680, 133, 826, 843, 747, 753, 79, 82, 593, 9, 11, 792, 715, 212, 174, 171, 44, 564, 741, 764]
Number of reactions after removing duplicates: 849


In [ ]:
def extract_unique_smiles(lagom_df):

    # Combine SMILES from both drug and metabolite columns
    combined = pd.concat([lagom_df["drug"], lagom_df["metabolite"]], ignore_index=True)

    # Remove duplicate SMILES strings
    unique_smiles = combined.drop_duplicates()

    # Convert to DataFrame
    df = unique_smiles.to_frame(name="smiles")

    print(f"Unique SMILES in LAGOM: {len(df)}")

    return df


df = pd.read_csv("inbetween_data/matched_reactions_deduplicated.csv")
df = extract_unique_smiles(df)
df.to_csv("unique_smiles.csv", index=False)

Unique SMILES in LAGOM: 372
